# Lab: give an agent four memory layers

An agent that forgets everything when the conversation ends is a goldfish. In this lab a real agent (DeepSeek and an embedding model, through OpenRouter) gets **four memory layers**. You fill in eight blanks (`___`) in three small functions, one per idea, and watch what each layer is good at, where it fails, and how to **delete** it all on request.

| Layer | Holds | Found by | Lives |
|---|---|---|---|
| 1 scratchpad | the agent's "Thought:" line | it is in the prompt | one run |
| 2 buffer | recent messages + a summary of older ones | the whole buffer | one conversation (thread) |
| 3 notes | free-text notes ("I loved the Lisbon harbour restaurant") | **similarity** search | one user, approximate |
| 4 profile | exact facts (`birthday: 15 March`) | **key** | one user, exact |

**Setup:** put `OPENROUTER_API_KEY=...` in a `.env` file (see `.env.example`); never commit it. Run the cells in order. `lab.py` is given code (the model, the graph, the helpers, the tests): no need to read it. A test cell under each function tells you which blank is wrong.

In [ ]:
import lab                                # given: the agent, the model calls, the helpers, the tests

lab.check_setup()

## The agent's graph

Every turn goes through `recall` (read the memory and build the context), `respond` (call the model) and `write_memory` (update the layers). A message starting with `/forget` goes to `confirm_forget` instead, which pauses for a human before `purge` deletes. The **checkpointer** saves the state of each thread (layer 2); the **store** holds the notes and the profile of each user (layers 3 and 4).

In [ ]:
lab.show_graph(lab.build_agent().app)

## 1. A goldfish, then an agent with memory

Albert tells the agent three things in conversation A (a *thread*): a birthday, a correction, and a nickname. The next day, Albert opens a **new conversation B** and asks for the birthday. The same script runs twice: first with **no memory layer read**, then with **all layers**. The agent below uses simple given versions of layers 2 and 4 (keep every message, overwrite a value); you write better ones next.

In [ ]:
script = ["Hello, I am Albert. My birthday is 14 March.", "Sorry, I misspoke: my birthday is 15 March.",
          "Please call me Bertie, and I am allergic to peanuts."]

for label, layers in (("NO MEMORY", set()), ("ALL LAYERS", None)):
    print(f"=== {label} ===")
    agent = lab.build_agent(layers=layers)
    for text in script:
        lab.say(agent, text, thread="A")
    lab.say(agent, "When is my birthday?", thread="B", show_context=layers is None)
    print()

Without memory, a new conversation starts empty: layer 2, the buffer, belongs to **one thread**. With layers 3 and 4, which belong to the **user**, the new conversation already knows Albert, and the context block shows why: the exact facts come first, labelled authoritative.

## 2. Layer 2: fill in `consolidate_buffer`

A long conversation cannot be sent in full at every turn. Keep the last `keep` messages as they are, and fold everything older into a running **summary**. `summarize(summary, old_turns)` is given: it calls the model once. Replace the two `___`.

In [ ]:
def consolidate_buffer(turns, summary, keep, summarize):
    """Return (recent_turns, new_summary)."""
    if len(turns) <= keep:
        return turns, summary            # nothing leaves the window: change nothing, call nothing
    old = turns[:-keep]                  # the messages that leave the window
    recent = turns[___:]                 # blank 1: the last `keep` messages
    new_summary = ___(summary, old)      # blank 2: the function that folds `old` into the summary (ONE call)
    return recent, new_summary

In [ ]:
lab.check_consolidate(consolidate_buffer)      # no model: a fake summarizer is used

## 3. Layer 4: fill in `upsert`

An exact fact has **one record per key**. When the value changes, the new value replaces the old one, and the old one is kept in `history`. It lives in the LangGraph **store**, under the namespace `("users", user_id, "profile")`. Useful calls: `store.get(namespace, key)` (returns an item with `.value`, or `None`) and `store.put(namespace, key, value_dict)`. Replace the three `___`.

In [ ]:
def upsert(store, user_id, key, value):
    """Create or update one fact: {"value": value, "history": [older values]}."""
    ns = ("users", user_id, "profile")
    item = store.get(ns, key)
    history = list(item.value["history"]) if item else []
    if item and item.value["value"] != ___:      # blank 3: the new value
        history.append(item.value["___"])        # blank 4: the field that holds the old value
    store.put(ns, ___, {"value": value, "history": history})    # blank 5: one record per ... what?

In [ ]:
lab.check_upsert(upsert)

## 4. Layer 3 versus layer 4

Suppose the agent stored these four notes over the months (layer 3), and the profile holds the current birthday (layer 4). The question: *"When is my birthday?"* Layer 3 is the store's **semantic index**: it ranks notes by cosine similarity of their embeddings.

In [ ]:
store = lab.build_agent().store
for i, text in enumerate(lab.BIRTHDAY_NOTES):
    store.put(("users", "albert", "notes"), str(i), {"text": text})

print("layer 3, search 'When is my birthday?':")
for hit in store.search(("users", "albert", "notes"), query="When is my birthday?", limit=4):
    print(f"  {hit.score:.2f}  {hit.value['text']}")

upsert(store, "albert", "birthday", "14 March"); upsert(store, "albert", "birthday", "15 March")
print("\nlayer 4, get 'birthday':", store.get(("users", "albert", "profile"), "birthday").value["value"])

Layer 3 returns a **ranked list of plausible notes**: the sister's birthday sits right behind, and the old 14 March is still there. Layer 4 returns **the answer**, exactly as stored. Rule: one right answer under a known key (a birthday, a name, an allergy) goes to layer 4; fuzzy text found by meaning goes to layer 3.

## 5. A long conversation

A new agent, 24 messages in one thread: three facts first (birthday, nickname, an allergy), then small talk. The window keeps only the last 6 messages. `feed` replays the messages (about a minute, one dot per message).

In [ ]:
agent = lab.build_agent(consolidate_buffer, upsert, keep=6)
lab.feed(agent, lab.CONVERSATION, thread="long")
lab.show_memory(agent, thread="long")

## 6. Which layer answers?

The same three questions, asked in a **new conversation** (the next day), with only some layers switched on. One row asks in the **same thread** as the 24 messages instead. `True` means the reply contains the right answer.

In [ ]:
lab.which_layer_answers(agent)

Read the table. The profile answers everything, in any thread. The summary only helps in its own thread. The notes are approximate: they may or may not surface the right statement, and short statements ("I am allergic to peanuts.") never become notes, because only statements of six words or more do.

## 7. Deletion: fill in `forget_user`

A user asks to be forgotten. Every place that holds their data must be emptied, and you must be able to **show** it. In this agent that is two namespaces of the store: `("users", user_id, "profile")` and `("users", user_id, "notes")`. `store.search(namespace, limit=1000)` lists the items (each has `.key`); `store.delete(namespace, key)` removes one. Replace the three `___`.

In [ ]:
def forget_user(store, user_id):
    """Delete everything stored about `user_id`. Return the counts: {"notes": n, "facts": m}."""
    report = {}
    for kind, name in (("notes", "notes"), ("profile", "facts")):
        ns = ("users", ___, kind)            # blank 6: whose data? (another user's must not change)
        items = store.search(ns, limit=1000)
        for item in items:
            store.___(ns, item.key)          # blank 7: remove one item
        report[name] = ___                   # blank 8: how many were removed
    return report

In [ ]:
lab.check_forget(forget_user)

## 8. Forget me

The message `/forget` goes through an approval step with `interrupt()`: the **pause of Session 3**, now guarding a deletion. It shows **what will be deleted**. First say no, then say yes.

In [ ]:
agent = lab.build_agent(consolidate_buffer, upsert, forget_user)
lab.feed(agent, lab.CONVERSATION[:3] + ["I loved the seafood restaurant near the Lisbon harbour."], thread="A")
lab.show_memory(agent)
print()
lab.say(agent, "/forget", thread="F1")
lab.resume(agent, "F1", approve=False)
lab.show_memory(agent)

In [ ]:
lab.say(agent, "/forget", thread="F2")
lab.resume(agent, "F2", approve=True)
lab.show_memory(agent)
print()
lab.say(agent, "When is my birthday?", thread="tomorrow");

Nothing left in any layer: the agent does not know Albert any more.

## Questions

1. A user says "I'm vegetarian" in January and "I eat fish now" in March. Which layer holds it, what does your `upsert` do, and what would layer 3 return?
2. Name two reasons a summary cannot be your only long-term memory. Where else, besides the store, can Albert's data still sit after `forget_user`?
3. For 10,000 users, would you keep one search index per user or one shared index with a filter? What do you gain and lose?